
# Karlsson et al. 2023 OmniLog dataset: parser and master `.pkl` builder

This notebook organizes the raw data from **Karlsson et al. 2023, _The power of light: Impact on the performance of biocontrol agents under minimal nutrient conditions_** into a single analysis-ready Python master file.

The notebook does four things:

1. Summarizes the dataset structure based on the files in the uploaded folder/archive.
2. Parses every OmniLog plate block from the raw CSV files.
3. Extracts raw time courses for all samples/wells, together with metadata.
4. Saves a master `.pkl` file with clear keys for downstream extraction.

Important: these are **OmniLog respiration/color-change trajectories**, not OD600 growth curves. The raw values are OmniLog units generated from tetrazolium dye reduction due to respiration.



## Data summary from the uploaded folder

The archive/folder contains six raw kinetic CSV files, one for each light treatment:

| File | Treatment | Observed plate blocks | Time rows per block | Within-block time span | Notes |
|---|---:|---:|---:|---:|---|
| `SLF_Darkness_allHours_kinetic.csv` | darkness | 36 | 385 | 0–96 h | Continuous OmniLog measurement every 15 min |
| `SLF_420_allhours_kinetic.csv` | 420 nm | 240 | 5 | 0–1 h per measurement window | Manual 1-hour OmniLog read windows; 8 setup-time windows observed |
| `SLF_450_allHours_kinetic.csv` | 450/460 nm | 320 | 5 | 0–1 h per measurement window | Manual 1-hour OmniLog read windows; 10 setup-time windows observed |
| `SLF_530_allHours_kinetic.csv` | 530 nm | 320 | 5 | 0–1 h per measurement window | Manual 1-hour OmniLog read windows; 10 setup-time windows observed |
| `SLF_620_allHours_kinetic.csv` | 620/630 nm | 310 | 5 | 0–1 h per measurement window | Manual 1-hour OmniLog read windows; 10 setup-time windows observed; one expected Ba/PM1 replicate block set is absent in the file |
| `SLF_660_allHours_kinetic.csv` | 660 nm | 320 | 5 | 0–1 h per measurement window | Manual 1-hour OmniLog read windows; 10 setup-time windows observed |

Organisms encoded in sample names:

| Sample prefix | Organism |
|---|---|
| `Ba` | *Bacillus amyloliquefaciens* |
| `Pchl` / `Pc` | *Pseudomonas chlororaphis* |
| `Sg` | *Streptomyces griseoviridis* |

Plate types:

- PM1 and PM2 phenotypic microarray plates.
- The raw files use slightly inconsistent strings, for example `PM 1-`, `PM 2-`, and `PM 2-A`; this notebook standardizes them to `PM1` and `PM2` while preserving the raw string.

Replicate/sample caveats observed directly in the files:

- The README says six replicates for each bacterium/treatment. In the raw light-treatment files, *Streptomyces griseoviridis* is observed as replicates 1–4 only. The darkness file contains Sg replicates 1–6.
- `SLF_420_allhours_kinetic.csv` has inconsistent raw sample labels such as `Ba1420`, `Ba1 420`, `Ba 5 420`, and `Pchl1460`-style labels. The parser normalizes these labels before extracting organism and replicate.
- In the 420 nm file, the expected Ba replicate/plate combinations `Ba4 PM1` and `Ba6 PM2` are absent as plate-block sets.
- In the 620/630 nm file, the expected `Ba2 PM1` block set is absent.
- The light-treatment files should be treated as **windowed/sparse time-course measurements**, not as independent 0–1 h growth curves. The notebook reconstructs absolute time as:


$\text{absolute hour actual} = \text{elapsed hours from earliest setup time within treatment} + \text{within-window hour}$

This preserves the observed timing in the files instead of forcing nominal/manual sampling times.



## Master `.pkl` structure

The output file `karlsson_master.pkl` is a dictionary with the following top-level keys:

```python
master = {
    "metadata": {...},
    "tables": {
        "plate_blocks": plate_blocks_df,
        "measurements_long": measurements_long_df,
        "trajectories": trajectories_df,
    },
    "indexes": {
        "by_treatment": {...},
        "by_organism": {...},
        "by_plate_type": {...},
        "by_treatment_organism": {...},
        "by_treatment_organism_plate": {...},
    },
    "helpers": {
        "note": "Use the helper functions in this notebook after loading the pkl."
    }
}
```

The most important table is `trajectories`, where each row corresponds to one reconstructed time course for a specific treatment × organism × replicate × PM plate × well.


In [ ]:

from pathlib import Path
import zipfile
import csv
import json
import re
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
# Put the path to the folder that contains the Karlsson raw data files.
# This folder should contain files such as:
#   SLF_Darkness_allHours_kinetic.csv
#   SLF_420_allhours_kinetic.csv
#   SLF_450_allHours_kinetic.csv
#   ...
#
# Examples:
#   DATA_DIR = Path("./karlsson_raw")
#   DATA_DIR = Path("/Users/your_name/path/to/karlsson_raw")
#   DATA_DIR = Path("../data/karlsson_raw")

DATA_DIR = Path("analyses/karlsson/data/raw") 

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"DATA_DIR does not exist: {DATA_DIR.resolve()}\n"
        "Please edit DATA_DIR so it points to the folder containing the raw Karlsson CSV files."
    )

if not DATA_DIR.is_dir():
    raise NotADirectoryError(
        f"DATA_DIR is not a folder: {DATA_DIR.resolve()}"
    )

# Output files will be written here.
OUTPUT_DIR = Path("analyses/karlsson/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find the raw OmniLog kinetic CSV files.
csv_paths = sorted(DATA_DIR.glob("SLF_*kinetic.csv"))

# A slightly more permissive fallback, in case capitalization differs.
if not csv_paths:
    csv_paths = sorted(DATA_DIR.glob("SLF_*Kinetic.csv"))

# A recursive fallback, in case the CSVs are inside subfolders.
if not csv_paths:
    csv_paths = sorted(DATA_DIR.rglob("SLF_*kinetic.csv"))

if not csv_paths:
    raise FileNotFoundError(
        f"No SLF_*kinetic.csv files found in {DATA_DIR.resolve()}.\n"
        "Please check that DATA_DIR points to the folder containing the raw OmniLog CSV files."
    )

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Found {len(csv_paths)} raw CSV files:")
for p in csv_paths:
    print(" -", p.relative_to(DATA_DIR) if p.is_relative_to(DATA_DIR) else p.name)

In [ ]:

# ---------------------------------------------------------------------
# Core parsing utilities
# ---------------------------------------------------------------------

def read_text(path: Path) -> str:
    """Read text using common encodings seen in these files."""
    for enc in ["utf-8-sig", "utf-8", "latin1", "cp1252"]:
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError:
            continue
    return path.read_text(errors="replace")


def treatment_from_filename(filename: str) -> str:
    """Return standardized light treatment name from raw file name."""
    s = filename.lower()
    if "dark" in s:
        return "darkness"
    if "420" in s:
        return "420"
    if "450" in s:
        return "450/460"
    if "530" in s:
        return "530"
    if "620" in s:
        return "620/630"
    if "660" in s:
        return "660"
    return "unknown"


def standardize_plate_type(raw_plate_type: str) -> str:
    """Standardize PM plate strings while preserving raw plate type separately."""
    s = (raw_plate_type or "").replace(" ", "").upper()
    if s.startswith("PM1"):
        return "PM1"
    if s.startswith("PM2"):
        return "PM2"
    return "unknown"


def parse_setup_time(raw_setup_time: str):
    """Parse setup time strings such as 'sep 02 2019 2:11 AM'."""
    if raw_setup_time is None or str(raw_setup_time).strip() == "":
        return pd.NaT
    s = str(raw_setup_time).strip()
    # Pandas handles these well, but exact formats avoid some ambiguity.
    for fmt in ["%b %d %Y %I:%M %p", "%B %d %Y %I:%M %p"]:
        try:
            return pd.Timestamp(datetime.strptime(s, fmt))
        except ValueError:
            pass
    return pd.to_datetime(s, errors="coerce")


def parse_sample_number(sample_number: str):
    """
    Normalize raw sample labels and extract organism/replicate.

    Examples handled:
    - Badark1, Ba1 420, Ba1420, Ba 5 420
    - Pchldark1, Pchl1460, Pchl2 460
    - Sgdark6, Sg1 420, Sg 2 420
    """
    raw = "" if sample_number is None else str(sample_number)
    clean = re.sub(r"\s+", "", raw)
    low = clean.lower()

    if low.startswith("pchl"):
        prefix = "Pchl"
        organism = "Pseudomonas chlororaphis"
        rest = clean[4:]
    elif low.startswith("pc"):
        prefix = "Pc"
        organism = "Pseudomonas chlororaphis"
        rest = clean[2:]
    elif low.startswith("ba"):
        prefix = "Ba"
        organism = "Bacillus amyloliquefaciens"
        rest = clean[2:]
    elif low.startswith("sg"):
        prefix = "Sg"
        organism = "Streptomyces griseoviridis"
        rest = clean[2:]
    else:
        prefix = "unknown"
        organism = "unknown"
        rest = clean

    # Replicates are encoded as the first digit 1-6 after the organism prefix.
    m = re.search(r"([1-6])", rest)
    replicate = int(m.group(1)) if m else np.nan

    return {
        "sample_number_raw": raw,
        "sample_number_clean": clean,
        "sample_prefix": prefix,
        "organism": organism,
        "replicate": replicate,
    }


def parse_omnilog_csv(path: Path):
    """
    Parse one OmniLog CSV file into:
    1. plate_blocks: one row per plate block
    2. measurements_long: one row per block x time x well value
    """
    treatment = treatment_from_filename(path.name)
    text = read_text(path)
    lines = text.splitlines()

    block_records = []
    measurement_frames = []

    i = 0
    block_i = 0
    while i < len(lines):
        if not lines[i].startswith("Data File"):
            i += 1
            continue

        # Metadata until the Hour header.
        meta = {}
        j = i
        while j < len(lines) and not lines[j].lstrip().startswith("Hour"):
            parts = next(csv.reader([lines[j]]))
            if parts:
                key = parts[0].strip()
                val = parts[1].strip() if len(parts) > 1 else ""
                if key:
                    meta[key] = val
            j += 1

        if j >= len(lines):
            break

        header = [h.strip() for h in next(csv.reader([lines[j]]))]
        if not header or header[0].lower() != "hour":
            i = j + 1
            continue

        data_rows = []
        k = j + 1
        while k < len(lines):
            line = lines[k]
            if not line.strip() or set(line.strip()) == {","} or line.startswith("Data File"):
                break
            row = [x.strip() for x in next(csv.reader([line]))]
            if not row or not row[0]:
                break
            data_rows.append(row)
            k += 1

        well_cols = [h for h in header[1:] if h]
        block_id = f"{treatment.replace('/', '_')}_{path.stem}_{block_i:04d}"
        sample_info = parse_sample_number(meta.get("Sample Number", ""))
        setup_time_raw = meta.get("Set up Time", "")
        setup_ts = parse_setup_time(setup_time_raw)
        plate_type_raw = meta.get("Plate Type", "")
        plate_type = standardize_plate_type(plate_type_raw)

        block_record = {
            "block_id": block_id,
            "source_file": path.name,
            "treatment": treatment,
            "setup_time_raw": setup_time_raw,
            "setup_time": setup_ts,
            "position": meta.get("Position", ""),
            "plate_type_raw": plate_type_raw,
            "plate_type": plate_type,
            "strain_type_raw": meta.get("Strain Type", ""),
            "strain_name_raw": meta.get("Strain Name", ""),
            "strain_number_raw": meta.get("Strain Number", ""),
            "other_raw": meta.get("Other", ""),
            "data_file_raw": meta.get("Data File", ""),
            "n_time_rows": len(data_rows),
            "n_wells": len(well_cols),
        }
        block_record.update(sample_info)

        # Convert block table to long format.
        if data_rows:
            block_df = pd.DataFrame(data_rows, columns=header[:len(data_rows[0])])
            block_df = block_df.rename(columns={block_df.columns[0]: "within_window_hour"})
            block_df["within_window_hour"] = pd.to_numeric(block_df["within_window_hour"], errors="coerce")
            for c in well_cols:
                block_df[c] = pd.to_numeric(block_df[c], errors="coerce")

            long_df = block_df.melt(
                id_vars=["within_window_hour"],
                value_vars=well_cols,
                var_name="well",
                value_name="omnilog_value",
            )
            for key, val in block_record.items():
                long_df[key] = val
            measurement_frames.append(long_df)

            block_record["hour_min"] = float(block_df["within_window_hour"].min())
            block_record["hour_max"] = float(block_df["within_window_hour"].max())
        else:
            block_record["hour_min"] = np.nan
            block_record["hour_max"] = np.nan

        block_records.append(block_record)
        block_i += 1
        i = max(k, j + 1)

    plate_blocks = pd.DataFrame(block_records)
    measurements_long = pd.concat(measurement_frames, ignore_index=True) if measurement_frames else pd.DataFrame()
    return plate_blocks, measurements_long


In [ ]:

# ---------------------------------------------------------------------
# Parse all raw CSV files
# ---------------------------------------------------------------------
plate_block_tables = []
measurement_tables = []

for path in csv_paths:
    print(f"Parsing {path.name} ...")
    blocks_df, long_df = parse_omnilog_csv(path)
    plate_block_tables.append(blocks_df)
    measurement_tables.append(long_df)

plate_blocks_df = pd.concat(plate_block_tables, ignore_index=True)
measurements_long_df = pd.concat(measurement_tables, ignore_index=True)

# Compute actual absolute timing within each treatment from setup timestamps.
# For darkness, all setup times are the same and within_window_hour already spans 0-96 h.
# This is vectorized because measurements_long_df has ~2 million rows.
min_setup_by_treatment = plate_blocks_df.groupby("treatment")["setup_time"].min().to_dict()
measurements_long_df["treatment_start_time"] = measurements_long_df["treatment"].map(min_setup_by_treatment)
measurements_long_df["setup_elapsed_hour"] = (
    (measurements_long_df["setup_time"] - measurements_long_df["treatment_start_time"]).dt.total_seconds() / 3600
)
measurements_long_df["absolute_hour_actual"] = measurements_long_df["setup_elapsed_hour"] + measurements_long_df["within_window_hour"]

# Add row/column indices for wells.
measurements_long_df["well_row"] = measurements_long_df["well"].str.extract(r"([A-H])", expand=False)
measurements_long_df["well_col"] = pd.to_numeric(measurements_long_df["well"].str.extract(r"(\d+)", expand=False), errors="coerce").astype("Int64")

print("Parsed plate blocks:", plate_blocks_df.shape)
print("Parsed long measurements:", measurements_long_df.shape)

display(plate_blocks_df.head())
display(measurements_long_df.head())


In [ ]:

# ---------------------------------------------------------------------
# Sanity-check summaries
# ---------------------------------------------------------------------
file_summary = (
    plate_blocks_df
    .groupby(["source_file", "treatment"], dropna=False)
    .agg(
        n_plate_blocks=("block_id", "nunique"),
        n_raw_samples=("sample_number_raw", "nunique"),
        n_clean_samples=("sample_number_clean", "nunique"),
        organisms=("organism", lambda x: "; ".join(sorted(set(map(str, x))))),
        plate_types=("plate_type", lambda x: "; ".join(sorted(set(map(str, x))))),
        time_rows_per_block=("n_time_rows", lambda x: sorted(set(x))),
        within_hour_max=("hour_max", lambda x: sorted(set(x))),
    )
    .reset_index()
)

display(file_summary)

window_summary = (
    plate_blocks_df
    .dropna(subset=["setup_time"])
    .assign(treatment_start_time=lambda d: d["treatment"].map(min_setup_by_treatment),
            setup_elapsed_hour=lambda d: (d["setup_time"] - d["treatment_start_time"]).dt.total_seconds() / 3600)
    .groupby("treatment")
    .agg(
        n_setup_windows=("setup_elapsed_hour", "nunique"),
        setup_elapsed_hours=("setup_elapsed_hour", lambda x: [round(v, 2) for v in sorted(set(x))]),
    )
    .reset_index()
)

display(window_summary)


In [ ]:

# ---------------------------------------------------------------------
# Identify missing expected plate/replicate combinations in the raw files
# ---------------------------------------------------------------------
# This cell does not change the data. It reports observed absences so that
# downstream analyses do not silently assume a complete balanced design.

expected_light_reps = {
    "Bacillus amyloliquefaciens": range(1, 7),
    "Pseudomonas chlororaphis": range(1, 7),
    "Streptomyces griseoviridis": range(1, 5),  # observed in light-treatment raw files
}
expected_dark_reps = {
    "Bacillus amyloliquefaciens": range(1, 7),
    "Pseudomonas chlororaphis": range(1, 7),
    "Streptomyces griseoviridis": range(1, 7),
}
expected_plates = ["PM1", "PM2"]

missing_records = []
for treatment, reps_by_org in [("darkness", expected_dark_reps)]:
    sub = plate_blocks_df[plate_blocks_df["treatment"] == treatment]
    for org, reps in reps_by_org.items():
        for rep in reps:
            for plate in expected_plates:
                n = len(sub[(sub["organism"] == org) & (sub["replicate"] == rep) & (sub["plate_type"] == plate)])
                if n == 0:
                    missing_records.append({"treatment": treatment, "organism": org, "replicate": rep, "plate_type": plate})

for treatment in ["420", "450/460", "530", "620/630", "660"]:
    sub = plate_blocks_df[plate_blocks_df["treatment"] == treatment]
    for org, reps in expected_light_reps.items():
        for rep in reps:
            for plate in expected_plates:
                n = len(sub[(sub["organism"] == org) & (sub["replicate"] == rep) & (sub["plate_type"] == plate)])
                if n == 0:
                    missing_records.append({"treatment": treatment, "organism": org, "replicate": rep, "plate_type": plate})

missing_design_df = pd.DataFrame(missing_records)
display(missing_design_df)


In [ ]:

# ---------------------------------------------------------------------
# Build reconstructed trajectories
# ---------------------------------------------------------------------
# One trajectory = one treatment x organism x replicate x plate_type x well.
# The trajectory keeps all raw measured time points and values.

TRAJECTORY_GROUP_COLS = [
    "treatment",
    "organism",
    "replicate",
    "plate_type",
    "well",
]

# Sort once globally rather than sorting each group independently.
measurements_for_grouping = measurements_long_df.sort_values(
    TRAJECTORY_GROUP_COLS + ["absolute_hour_actual", "within_window_hour", "block_id"],
    kind="mergesort",  # stable sort
)

# Keep source block IDs and raw labels as provenance.
def _unique_sorted(values):
    return sorted(set(v for v in values if pd.notna(v)))

trajectory_records = []
for key, g in measurements_for_grouping.groupby(TRAJECTORY_GROUP_COLS, dropna=False, sort=False):
    treatment, organism, replicate, plate_type, well = key
    rep_label = int(replicate) if pd.notna(replicate) else "NA"
    trajectory_id = f"{treatment}|{organism}|rep{rep_label}|{plate_type}|{well}"
    times = g["absolute_hour_actual"].to_numpy(dtype=float)
    values = g["omnilog_value"].to_numpy(dtype=float)
    trajectory_records.append({
        "trajectory_id": trajectory_id,
        "treatment": treatment,
        "organism": organism,
        "replicate": replicate,
        "plate_type": plate_type,
        "well": well,
        "well_row": g["well_row"].iloc[0],
        "well_col": int(g["well_col"].iloc[0]) if pd.notna(g["well_col"].iloc[0]) else np.nan,
        "n_points": len(g),
        "time_hours": times,
        "within_window_hours": g["within_window_hour"].to_numpy(dtype=float),
        "values": values,
        "block_ids": _unique_sorted(g["block_id"]),
        "source_files": _unique_sorted(g["source_file"]),
        "raw_sample_numbers": _unique_sorted(g["sample_number_raw"]),
        "is_continuous_darkness": bool(treatment == "darkness"),
        "is_windowed_light": bool(treatment != "darkness"),
        "time_min": float(np.nanmin(times)),
        "time_max": float(np.nanmax(times)),
    })

trajectories_df = pd.DataFrame(trajectory_records)
print("Trajectories:", trajectories_df.shape)
display(trajectories_df.head())

display(
    trajectories_df.groupby(["treatment", "organism", "plate_type"])
    .agg(n_trajectories=("trajectory_id", "nunique"), n_points_values=("n_points", lambda x: sorted(set(x))))
    .reset_index()
)


In [ ]:

# ---------------------------------------------------------------------
# Build indexes for convenient extraction from the master pkl
# ---------------------------------------------------------------------

def build_index(df, cols):
    out = {}
    for key, sub in df.groupby(cols, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        out[key] = sub["trajectory_id"].tolist()
    return out

indexes = {
    "by_treatment": build_index(trajectories_df, ["treatment"]),
    "by_organism": build_index(trajectories_df, ["organism"]),
    "by_plate_type": build_index(trajectories_df, ["plate_type"]),
    "by_treatment_organism": build_index(trajectories_df, ["treatment", "organism"]),
    "by_treatment_organism_plate": build_index(trajectories_df, ["treatment", "organism", "plate_type"]),
}

# Example keys
for name, idx in indexes.items():
    print(name, "example keys:", list(idx.keys())[:5])


In [ ]:

# ---------------------------------------------------------------------
# Optional: create fixed-length 128-point arrays for downstream models
# ---------------------------------------------------------------------
# This function does NOT overwrite the raw trajectories. It creates a separate
# array representation by interpolation. Use with care for light treatments,
# because they are sparse/windowed measurements.


def make_fixed_length_array(
    trajectories: pd.DataFrame,
    n_points: int = 128,
    time_min: float = 0.0,
    time_max: float = 96.0,
    normalize: str | None = None,
):
    """
    Convert trajectories to a fixed-length array by linear interpolation.

    Parameters
    ----------
    trajectories : DataFrame
        Usually a subset of trajectories_df.
    n_points : int
        Number of output time points.
    time_min, time_max : float
        Common grid bounds in hours.
    normalize : None, 'max', or 'minmax'
        Optional per-trajectory normalization.

    Returns
    -------
    fm_ready : dict with X, time_grid, labels, trajectory_ids.
    """
    time_grid = np.linspace(time_min, time_max, n_points)
    X = []
    labels = []
    trajectory_ids = []
    normalization_records = []

    for _, row in trajectories.iterrows():
        t = np.asarray(row["time_hours"], dtype=float)
        y = np.asarray(row["values"], dtype=float)
        ok = np.isfinite(t) & np.isfinite(y)
        t = t[ok]
        y = y[ok]
        if len(t) < 2:
            continue

        # Average duplicated time points if present.
        tmp = pd.DataFrame({"t": t, "y": y}).groupby("t", as_index=False)["y"].mean().sort_values("t")
        t = tmp["t"].to_numpy()
        y = tmp["y"].to_numpy()

        # Interpolate only within observed range; outside range becomes NaN.
        yi = np.interp(time_grid, t, y)
        yi[(time_grid < t.min()) | (time_grid > t.max())] = np.nan

        norm_info = {"trajectory_id": row["trajectory_id"], "normalization": normalize}
        if normalize == "max":
            max_val = np.nanmax(yi)
            norm_info["max_value"] = float(max_val)
            if np.isfinite(max_val) and max_val != 0:
                yi = yi / max_val
        elif normalize == "minmax":
            mn = np.nanmin(yi)
            mx = np.nanmax(yi)
            norm_info["min_value"] = float(mn)
            norm_info["max_value"] = float(mx)
            if np.isfinite(mx - mn) and (mx - mn) != 0:
                yi = (yi - mn) / (mx - mn)
        elif normalize is not None:
            raise ValueError("normalize must be None, 'max', or 'minmax'")

        X.append(yi.astype(np.float32))
        labels.append({
            "trajectory_id": row["trajectory_id"],
            "treatment": row["treatment"],
            "organism": row["organism"],
            "replicate": row["replicate"],
            "plate_type": row["plate_type"],
            "well": row["well"],
            "n_raw_points": row["n_points"],
        })
        trajectory_ids.append(row["trajectory_id"])
        normalization_records.append(norm_info)

    return {
        "X": np.vstack(X) if X else np.empty((0, n_points), dtype=np.float32),
        "time_grid": time_grid.astype(np.float32),
        "labels": pd.DataFrame(labels),
        "trajectory_ids": trajectory_ids,
        "normalization_info": pd.DataFrame(normalization_records),
        "interpolation_note": "Linear interpolation on actual observed hours; outside observed range is NaN before optional normalization.",
    }

# Example: darkness only, no normalization.
# darkness_fm_ready = make_fixed_length_array(
#     trajectories_df[trajectories_df["treatment"] == "darkness"],
#     n_points=128,
#     time_min=0,
#     time_max=96,
#     normalize=None,
# )
# darkness_fm_ready["X"].shape


In [ ]:

# ---------------------------------------------------------------------
# Assemble and save the master pkl
# ---------------------------------------------------------------------
metadata = {
    "dataset_name": "Karlsson et al. 2023 - The power of light",
    "measurement_type": "OmniLog respiration/color-change values from Biolog phenotypic microarrays",
    #"raw_zip": str(RAW_ZIP),
    "data_dir": str(DATA_DIR),
    "csv_files": [p.name for p in csv_paths],
    "organism_prefix_map": {
        "Ba": "Bacillus amyloliquefaciens",
        "Pchl/Pc": "Pseudomonas chlororaphis",
        "Sg": "Streptomyces griseoviridis",
    },
    "plate_type_note": "Raw PM plate type strings were standardized to PM1/PM2 in plate_type; original strings are retained in plate_type_raw.",
    "timing_note": (
        "Darkness is continuous from 0 to 96 h at 0.25 h resolution. "
        "Light treatments are 1-h OmniLog measurement windows. absolute_hour_actual is calculated as "
        "elapsed hours from the earliest setup time within each treatment plus within_window_hour."
    ),
    "sample_name_note": "Raw sample names have inconsistent spacing/formatting; sample_number_clean and parsed organism/replicate are provided.",
    "known_raw_file_caveats": [
        "Light-treatment Sg samples are observed as replicates 1-4 in the raw files; darkness contains Sg replicates 1-6.",
        "420 nm file lacks Ba4 PM1 and Ba6 PM2 plate-block sets as observed in the raw files.",
        "620/630 nm file lacks Ba2 PM1 plate-block set as observed in the raw files.",
    ],
}

master = {
    "metadata": metadata,
    "tables": {
        "plate_blocks": plate_blocks_df,
        "measurements_long": measurements_long_df,
        "trajectories": trajectories_df,
        "file_summary": file_summary,
        "window_summary": window_summary,
        "missing_design": missing_design_df,
    },
    "indexes": indexes,
    "helpers": {
        "note": "Helper functions are defined in the notebook. They are not pickled as executable functions for portability.",
        "trajectory_group_columns": TRAJECTORY_GROUP_COLS,
    },
}

PKL_PATH = OUTPUT_DIR / "karlsson_master.pkl"
with open(PKL_PATH, "wb") as f:
    pickle.dump(master, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved master pkl to: {PKL_PATH.resolve()}")
print(f"Approximate pkl size: {PKL_PATH.stat().st_size / 1024**2:.2f} MB")


In [ ]:

# ---------------------------------------------------------------------
# How to reload and extract datasets later
# ---------------------------------------------------------------------
with open(PKL_PATH, "rb") as f:
    loaded = pickle.load(f)

traj = loaded["tables"]["trajectories"]
long = loaded["tables"]["measurements_long"]

print(loaded.keys())
print(loaded["tables"].keys())

# Example 1: all darkness trajectories
ids_dark = loaded["indexes"]["by_treatment"][("darkness",)]
darkness_trajectories = traj[traj["trajectory_id"].isin(ids_dark)]
print("Darkness trajectories:", darkness_trajectories.shape)

# Example 2: all Bacillus trajectories under 530 nm
ids_ba_530 = loaded["indexes"]["by_treatment_organism"][("530", "Bacillus amyloliquefaciens")]
ba_530_trajectories = traj[traj["trajectory_id"].isin(ids_ba_530)]
print("Bacillus / 530 nm trajectories:", ba_530_trajectories.shape)

# Example 3: raw long-format values for one trajectory
one_id = ba_530_trajectories["trajectory_id"].iloc[0]
one_traj = traj[traj["trajectory_id"] == one_id].iloc[0]
print("Example trajectory id:", one_id)
print("First 10 times:", one_traj["time_hours"][:10])
print("First 10 values:", one_traj["values"][:10])



## Recommended downstream usage

For most analyses, use `master["tables"]["trajectories"]` rather than `measurements_long`:

- `trajectories` is compact and has one row per reconstructed time course.
- `measurements_long` is most useful when you need exact provenance for every raw measurement point.
- For foundation-model testing, use `make_fixed_length_array(...)` to create a separate 128-point matrix. Keep those arrays separate from the raw master data so the interpolation/normalization choices remain explicit.

Suggested conservative workflow:

1. Start with darkness trajectories because they are continuous 0–96 h time courses.
2. Treat light trajectories as sparse/windowed time courses.
3. For light treatments, compare analyses using actual observed times versus any 128-point interpolated representation.
4. Do not assume all organism × replicate × PM plate combinations are complete; use `missing_design_df` to track observed absences.


## Generated output files

The notebook generates several output files in the `outputs/` directory. These files are designed to serve different purposes: reproducible documentation, portable tabular data inspection, and convenient Python-native downstream analysis. Together, they preserve the raw OmniLog kinetic measurements, the parsed plate/block metadata, and reconstructed time-course trajectories.

### 1. `karlsson_master.pkl`

This is the main master file generated by the notebook. It is a Python pickle file containing a structured dictionary that organizes all parsed data and metadata from the Karlsson raw OmniLog files.

This file is intended to be the primary analysis-ready object for Python workflows. It preserves the hierarchy of the experiment while also providing convenient access to long-format measurements and reconstructed trajectories.

The master pickle contains several top-level keys:

```python
master.keys()
```

Expected major keys include:

```text
metadata
plate_blocks
measurements_long
trajectories
indexes
schema
```

#### `master["metadata"]`

This entry stores dataset-level information. It describes the source dataset, the type of measurement, the organisms, the light treatments, and important parsing assumptions.

Typical contents include:

```text
dataset_name
measurement_type
raw_files
light_treatments
organisms
plate_types
notes
```

The measurement type is OmniLog respiration/color-development signal, not OD600. This distinction is important because these time courses reflect metabolic activity on Biolog phenotype microarray plates rather than conventional optical-density growth curves.

#### `master["plate_blocks"]`

This entry contains a `pandas.DataFrame` with one row per parsed OmniLog plate block.

Each raw CSV file contains multiple plate blocks. A plate block corresponds to one plate measurement unit with associated metadata and a 96-well kinetic table.

This table is useful for checking whether the raw files were parsed correctly and for tracing each trajectory back to its source.

Important columns include:

```text
block_id
source_file
light_treatment
organism
replicate
sample_name_raw
sample_name_clean
plate_type
window_index
setup_time_raw
setup_time_hours
n_time_rows
n_wells
```

The exact column names may vary slightly depending on the parser version, but the conceptual structure should remain the same.

Important interpretation:

* For the darkness condition, each plate block typically contains a continuous 0–96 hour kinetic measurement.
* For light-treated samples, each plate block usually represents a short OmniLog measurement window after the plate had been incubated under a given light condition.
* Therefore, the light-treatment blocks should not be interpreted as independent full growth curves. They are pieces of a longer, windowed time course.

#### `master["measurements_long"]`

This entry contains the fully parsed raw measurement table in long format.

Each row corresponds to one measured value from one well at one time point within one plate block.

This is the most traceable representation of the raw data. It is larger than the metadata and trajectory tables because it contains every individual well-by-time measurement.

Important columns include:

```text
block_id
source_file
light_treatment
organism
replicate
plate_type
well
within_window_hour
absolute_hour
omnilog_value
```

Interpretation of time columns:

* `within_window_hour` is the time reported inside a specific OmniLog measurement block.
* `absolute_hour` is the reconstructed time relative to the beginning of the biological experiment, combining the setup/window time and the within-window measurement time.
* For darkness, `within_window_hour` and `absolute_hour` are usually equivalent or nearly equivalent because the full kinetic curve was measured continuously.
* For light treatments, `absolute_hour` is the more useful column for reconstructing the full time course.

This table is useful for:

* plotting raw measurements,
* checking individual wells,
* debugging parser behavior,
* reconstructing trajectories with different assumptions,
* exporting to non-Python workflows.

#### `master["trajectories"]`

This entry contains one row per reconstructed time course.

A trajectory is defined by grouping raw measurements by biological and experimental identity, typically:

```text
light_treatment
organism
replicate
plate_type
well
```

Each row stores the time and value arrays for one well-specific trajectory.

Important columns include:

```text
trajectory_id
light_treatment
organism
replicate
plate_type
well
time_hours
values
n_points
is_continuous
is_windowed
```

Interpretation:

* `time_hours` contains the reconstructed absolute time points.
* `values` contains the corresponding OmniLog values.
* `n_points` is the number of time points available for that trajectory.
* `is_continuous` indicates whether the trajectory came from continuous measurement, mainly the darkness condition.
* `is_windowed` indicates whether the trajectory was assembled from separate measurement windows, mainly the light-treatment conditions.

This table is the most useful object for downstream modeling because each row corresponds to one complete sample-level time course.

#### `master["indexes"]`

This entry contains convenience indexes for selecting subsets of the trajectory table.

These keys are designed to make it easier to extract specific datasets without manually filtering the entire table each time.

Typical index groups include:

```text
by_light_treatment
by_organism
by_plate_type
by_light_and_organism
by_light_organism_plate
```

Example usage:

```python
master = pd.read_pickle("outputs/karlsson_master.pkl")

trajectories = master["trajectories"]

# Get all darkness trajectories
idx = master["indexes"]["by_light_treatment"]["Darkness"]
darkness_df = trajectories.loc[idx]

# Get all Bacillus amyloliquefaciens trajectories
idx = master["indexes"]["by_organism"]["Bacillus amyloliquefaciens"]
bacillus_df = trajectories.loc[idx]

# Get a specific treatment-organism-plate subset
key = ("420", "Bacillus amyloliquefaciens", "PM1")
idx = master["indexes"]["by_light_organism_plate"][key]
subset_df = trajectories.loc[idx]
```

The exact key names depend on how the parser normalized treatment and organism labels. To inspect available keys, use:

```python
master["indexes"].keys()
```

and then:

```python
master["indexes"]["by_light_treatment"].keys()
master["indexes"]["by_organism"].keys()
master["indexes"]["by_plate_type"].keys()
```

#### `master["schema"]`

This entry stores a machine-readable description of the output structure. It explains the major tables, important columns, and parsing assumptions. It is included so that future users can understand the master pickle without relying only on the notebook.

---

### 2. `karlsson_trajectories.pkl`

This file contains only the reconstructed trajectory table.

It is a smaller and simpler alternative to `karlsson_master.pkl` when the goal is only to work with time courses.

Load it with:

```python
trajectories = pd.read_pickle("outputs/karlsson_trajectories.pkl")
```

Each row corresponds to one reconstructed trajectory, usually defined by:

```text
light_treatment × organism × replicate × plate_type × well
```

This file is useful for:

* plotting time courses,
* filtering by organism/treatment/plate/well,
* interpolating to 128 points,
* generating input arrays for machine-learning models.

Use `karlsson_master.pkl` when you need both data and metadata. Use `karlsson_trajectories.pkl` when you only need trajectory-level time courses.

---

### 3. `karlsson_measurements_long.csv.gz`

This is a compressed CSV version of the long-format raw measurement table.

It contains one row per measured value.

This file is larger than the trajectory table because it stores every raw time-point measurement separately.

It can be loaded with:

```python
measurements_long = pd.read_csv("outputs/karlsson_measurements_long.csv.gz")
```

This file is useful because it is portable and does not depend on Python pickle serialization. It can be opened in R, Python, command-line tools, or spreadsheet software after decompression.

Recommended uses:

* auditing raw parsed values,
* checking time reconstruction,
* rebuilding trajectories using alternative assumptions,
* sharing a transparent tabular version of the parsed data.

Because it is long-format, it is also ideal for `pandas`, `tidyverse`, and database-style operations.

---

### 4. `karlsson_plate_blocks.csv.gz`

This file contains one row per parsed OmniLog plate block.

It is a compressed CSV export of `master["plate_blocks"]`.

Load it with:

```python
plate_blocks = pd.read_csv("outputs/karlsson_plate_blocks.csv.gz")
```

This file is useful for checking the structure of the raw data before trajectory reconstruction.

Typical questions this file helps answer:

* How many plate blocks were parsed from each raw file?
* Which light treatment does each block correspond to?
* Which organism and replicate are associated with each block?
* How many time rows are present in each block?
* Which blocks correspond to continuous measurements versus short measurement windows?
* Are any blocks missing or unusually formatted?

This file is especially important for this dataset because the light-treatment files are windowed rather than continuous. The plate-block table helps document how the raw measurement windows were detected and assigned.

---

### 5. `karlsson_file_summary.csv.gz`

This file summarizes the parsed raw files at the file level.

It provides a compact overview of each input CSV file, including the inferred treatment and the number of parsed blocks.

Load it with:

```python
file_summary = pd.read_csv("outputs/karlsson_file_summary.csv.gz")
```

This file is useful for a quick sanity check after parsing.

Typical questions this file helps answer:

* Which raw files were found?
* How many plate blocks were detected in each file?
* Which treatment was inferred from each filename?
* Were all expected files parsed?

This is usually the first file to inspect when confirming that the parser worked correctly.

---

### 6. `karlsson_window_summary.csv.gz`

This file summarizes inferred measurement windows.

It is particularly useful for the light-treatment data, where time courses are assembled from multiple short OmniLog measurement windows rather than one continuous 96-hour measurement.

Load it with:

```python
window_summary = pd.read_csv("outputs/karlsson_window_summary.csv.gz")
```

This file helps document:

* the number of measurement windows per treatment,
* the inferred setup or absolute times of each window,
* whether all expected windows are present,
* whether some light treatments have missing or irregular windows.

This file is important because the windowed design is the main complication in the Karlsson dataset. It should be reviewed before using interpolated light-treatment trajectories for modeling.

---

### 7. `karlsson_missing_design.csv.gz`

This file summarizes missing or incomplete design combinations detected during parsing.

Load it with:

```python
missing_design = pd.read_csv("outputs/karlsson_missing_design.csv.gz")
```

The purpose of this file is to identify expected combinations that may be absent from the parsed dataset.

Potential missing combinations may involve:

```text
light_treatment
organism
replicate
plate_type
window_index
well
```

This file is useful for quality control.

Important interpretation:

* A missing entry does not always mean a parsing error.
* Some combinations may be biologically or experimentally absent.
* Some light-treatment files may have incomplete measurement windows.
* Some inconsistencies may arise from differences in raw sample naming or plate setup.

Before excluding data, inspect this file together with `karlsson_plate_blocks.csv.gz` and `karlsson_window_summary.csv.gz`.

---

### 8. `karlsson_master_schema.json`

This JSON file describes the organization of the master output files.

Load it with:

```python
import json

with open("outputs/karlsson_master_schema.json", "r") as f:
    schema = json.load(f)
```

This file is useful for documentation and reproducibility. It should describe:

* the purpose of each generated output file,
* the meaning of major tables,
* important column definitions,
* parsing assumptions,
* known caveats.

This file is especially useful if the data are shared with collaborators or revisited later.

---

## Recommended workflow

A typical downstream workflow should start with the master pickle:

```python
import pandas as pd

master = pd.read_pickle("outputs/karlsson_master.pkl")

plate_blocks = master["plate_blocks"]
measurements_long = master["measurements_long"]
trajectories = master["trajectories"]
indexes = master["indexes"]
```

For quick trajectory-level analysis:

```python
trajectories = pd.read_pickle("outputs/karlsson_trajectories.pkl")
```

For transparent data inspection outside Python pickle:

```python
measurements_long = pd.read_csv("outputs/karlsson_measurements_long.csv.gz")
plate_blocks = pd.read_csv("outputs/karlsson_plate_blocks.csv.gz")
window_summary = pd.read_csv("outputs/karlsson_window_summary.csv.gz")
```

For model input generation, use `trajectories`, not `measurements_long`, because `trajectories` already groups raw measurements into sample-level time courses.

---

## Important caveats

### Darkness versus light-treatment data

The darkness condition contains continuous OmniLog kinetic measurements over approximately 96 hours. These trajectories are straightforward to treat as continuous time courses.

The light-treatment data are different. They are assembled from short OmniLog measurement windows taken at different post-inoculation times. Therefore, each light-treatment time course should be interpreted as a reconstructed windowed trajectory.

This distinction matters for downstream modeling. The light-treatment trajectories can be interpolated to fixed-length 128-point curves, but the interpolation step should be interpreted as a preprocessing approximation rather than a direct measurement at all 128 time points.

### OmniLog signal is not OD600

The raw values are OmniLog respiration/color-development units from Biolog phenotype microarray plates. These values are related to metabolic activity on specific carbon sources, not direct biomass measurements.

Therefore, these trajectories should not be interpreted exactly like OD600 growth curves. They are still valuable microbial time-series data, but the biological meaning is closer to substrate-specific respiration/metabolic activity.

### Plate wells correspond to carbon-source conditions

Each PM plate contains 96 wells, and each well corresponds to a different substrate or control condition. Therefore, a trajectory grouped by `well` represents the organism’s OmniLog signal under a specific plate/well condition.

To interpret individual wells biologically, the PM1/PM2 well map should be joined to the trajectory table if substrate identities are needed.

### Pickle files are Python-specific

The `.pkl` files are convenient for Python because they preserve arrays and nested objects. However, they are less portable than CSV, JSON, or NPZ files.

For reproducibility, the notebook also exports compressed CSV and JSON files. These should be kept alongside the pickle files.

---

## Suggested file usage summary

| File                                | Main purpose                                | Recommended use                         |
| ----------------------------------- | ------------------------------------------- | --------------------------------------- |
| `karlsson_master.pkl`               | Complete Python-native master object        | Main analysis file                      |
| `karlsson_trajectories.pkl`         | Reconstructed trajectory table only         | Plotting and ML preprocessing           |
| `karlsson_measurements_long.csv.gz` | All raw parsed measurements in long format  | Auditing, sharing, alternative parsing  |
| `karlsson_plate_blocks.csv.gz`      | Metadata for each raw OmniLog block         | Quality control and parsing checks      |
| `karlsson_file_summary.csv.gz`      | Input-file-level summary                    | Quick parser sanity check               |
| `karlsson_window_summary.csv.gz`    | Summary of inferred measurement windows     | Checking light-treatment reconstruction |
| `karlsson_missing_design.csv.gz`    | Missing or incomplete expected combinations | Quality control                         |
| `karlsson_master_schema.json`       | Machine-readable schema/documentation       | Reproducibility and collaboration       |


# Karlsson dark trajectory preprocessing
The following cells perform the preprocessing on the **dark trajectories** from the Karlsson OmniLog dataset.

Workflow:

1. Load the Karlsson master `.pkl` generated by `Karlsson_master_pkl_builder.ipynb`.
2. Select dark trajectories.
3. Split each 385-point dark trajectory into three non-overlapping 128-point segments, dropping the final point.
4. Save a segment-level table that preserves and updates the original dark-trajectory labels.

In [ ]:
from pathlib import Path
import sys
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import r2_score, mean_squared_error

import torch
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

# Preferred location of the master pickle
PKL_PATH = Path("analyses/karlsson/outputs/karlsson_master.pkl")

# If the preferred path does not exist, search locally for the file.
if not PKL_PATH.exists():
    print(f"Preferred PKL file not found at: {PKL_PATH.resolve()}")
    print("Searching for karlsson_master.pkl in the current directory...")

    local_hits = []  # Avoid silently loading stale data from another directory.

    if local_hits:
        PKL_PATH = local_hits[0]
        print(f"Found master PKL at: {PKL_PATH.resolve()}")
    else:
        raise FileNotFoundError(
            "\nCould not find karlsson_master.pkl.\n\n"
            "Please run Karlsson_dark_preprocessing.ipynb first.\n"
            "The builder notebook should create:\n"
            "    analyses/karlsson/outputs/karlsson_master.pkl\n\n"
            "Also check that you are running this notebook from the "
            "correct project directory."
        )

# ---------------------------------------------------------------------
# Output directory
# ---------------------------------------------------------------------

OUTPUT_DIR = PKL_PATH.parent 
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loading master pkl from: {PKL_PATH.resolve()}")
print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")

# ---------------------------------------------------------------------
# Load master pickle
# ---------------------------------------------------------------------

try:
    with open(PKL_PATH, "rb") as f:
        master = pickle.load(f)
except Exception as e:
    raise RuntimeError(
        f"\nThe file was found, but could not be loaded:\n"
        f"    {PKL_PATH.resolve()}\n\n"
        f"Original error: {type(e).__name__}: {e}"
    ) from e

# ---------------------------------------------------------------------
# Validate master structure
# ---------------------------------------------------------------------

if not isinstance(master, dict):
    raise TypeError(
        f"\nExpected the pickle to contain a dictionary, "
        f"but found: {type(master).__name__}"
    )

if "tables" not in master:
    raise KeyError(
        "\nThe loaded master pickle does not contain the expected "
        "'tables' key.\n"
        f"Available top-level keys: {list(master.keys())}"
    )

print("Top-level keys:", list(master.keys()))
print("Tables:", list(master["tables"].keys()))
